<h2>File paths and imports</h2>

This steps is important because it tells the program where it can access the files needed throughout the process.

You must specify 3 paths:

<li> <b>model_path</b>: body detection model
<li> <b>input_video_directory</b>: Folder containing the input videos
<li> <b>output_video_directory</b>: Folder where the treated videos will be redirected to

You can also mention videos that you don't want to be treated. To do so, simply indicate their name(s) without the extension in <b>ignore_S1</b> and <b>ignore_S2</b>.

In [1]:
from ui_lib_strongSort import *
import torch
import os

# Enable cuDNN autotune for fixed-size video frames
torch.backends.cudnn.benchmark = True
# (Optional) improve matmul kernels on Ampere+
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

# Normalize device: use first visible GPU if available
device_str = "cuda:0" if torch.cuda.is_available() else "cpu"

# Path of the body detection model
body_model_path = "/home/ucl/ingi/trixen/ChimpRec/Code/Tracking/strong_sort/Body_detection_models/Body_detection_model.pt"

# Video paths
input_video_directory = "/home/ucl/ingi/trixen/ChimpRec/ChimpVideo/input"
output_video_directory = "/home/ucl/ingi/trixen/ChimpRec/ChimpVideo/Output"

ignore_S1 = ["20241019 - 14h29",
            "20241019 - 13h28",
            "20241015 - 12h41-treated"
            ]  # step 1

ignore_S2 = ["20241019 - 14h29",
             "20241019 - 13h28",
             "20241015 - 12h41-treated"
            ]  # step 2

ignore_S3 = ["20241019 - 14h29",
             "20241019 - 13h28",
             "20241015 - 12h41-treated"
        ]  # step 3

def has_audio_stream(video_path: str) -> bool:
    try:
        import subprocess, json
        probe_cmd = [
            "ffprobe", "-v", "error",
            "-select_streams", "a",
            "-show_entries", "stream=index",
            "-of", "json",
            video_path,
        ]
        out = subprocess.check_output(probe_cmd).decode("utf-8")
        data = json.loads(out)
        streams = data.get("streams", [])
        return len(streams) > 0
    except Exception:
        return True

# Directories
mannual_annotations_directory = f"{input_video_directory}/manual_annotations"
output_video_directory_temp = f"{output_video_directory}/temp"
raw_text_output_directory = f"{output_video_directory_temp}/raw_output"
treated_directory = f"{output_video_directory}/treated"
final_directory = f"{output_video_directory}/final"

for d in [
    input_video_directory,
    output_video_directory,
    mannual_annotations_directory,
    output_video_directory_temp,
    raw_text_output_directory,
]:
    os.makedirs(d, exist_ok=True)

# YOLOv8s initialization on GPU (if available) with FP16
YOLOv8s = YOLO(body_model_path)
YOLOv8s.to(device_str)
use_half = device_str.startswith("cuda")
# StrongSORT initialization (FP16 if on GPU)
reid_weights_path = "/home/ucl/ingi/trixen/ChimpRec/Code/Tracking/strong_sort/Re-ID_models/osnet_ain_x1_0_imagenet.pth"
configuration_file_path = None #"/home/ucl/ingi/trixen/ChimpRec/Code/Tracking/strong_sort/configs/strong_sort.yaml"
strongsort = build_strongsort(
    reid_weights=reid_weights_path,
    device=device_str,
    fp16=use_half,
    tracker_config_path=configuration_file_path,
)

2026-03-09 21:29:44.099 | MainProcess/MainThread | INFO     | /home/ucl/ingi/trixen/ChimpRec/.venv/lib/python3.10/site-packages/boxmot/trackers/basetracker.py:56 | __init__ - BaseTracker initialization parameters:
2026-03-09 21:29:44.099 | MainProcess/MainThread | INFO     | /home/ucl/ingi/trixen/ChimpRec/.venv/lib/python3.10/site-packages/boxmot/trackers/basetracker.py:57 | __init__ - det_thresh: 0.3
2026-03-09 21:29:44.099 | MainProcess/MainThread | INFO     | /home/ucl/ingi/trixen/ChimpRec/.venv/lib/python3.10/site-packages/boxmot/trackers/basetracker.py:58 | __init__ - max_age: 30
2026-03-09 21:29:44.099 | MainProcess/MainThread | INFO     | /home/ucl/ingi/trixen/ChimpRec/.venv/lib/python3.10/site-packages/boxmot/trackers/basetracker.py:59 | __init__ - max_obs: 50
2026-03-09 21:29:44.099 | MainProcess/MainThread | INFO     | /home/ucl/ingi/trixen/ChimpRec/.venv/lib/python3.10/site-packages/boxmot/trackers/basetracker.py:60 | __init__ - min_hits: 3
2026-03-09 21:29:44.099 | MainProc

<h2>First step:</h2>

This step will process the input videos automatically. In other words, it will draw rectangles around each individuals and track them throughout the video.

In [2]:
for input_video in os.listdir(input_video_directory):
    if not (input_video.endswith(".mp4") or input_video.endswith(".MP4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S1:
        print(f"{video_name}.mp4 ignored")
        continue

    annotation_file_path = f"{mannual_annotations_directory}/{video_name}.txt"
    try:
        with open(annotation_file_path, "x") as f:
            print(f"{video_name}.txt automatically created in {mannual_annotations_directory}.")
    except FileExistsError:
        print(f"{video_name}.txt already present in {mannual_annotations_directory}.")
    print()

    raw_txt_path = f"{raw_text_output_directory}/{video_name}.txt"
    perform_tracking(
        input_video_path=full_video_path,
        output_text_file_path=raw_txt_path,
        detection_model=YOLOv8s,
        tracker=strongsort,
        confidence_threshold=0.5,
        device=device_str,
        use_half=use_half,
    )
    print(f"Annotations ready for video: {full_video_path}.\n")

    processed_with_audio = f"{output_video_directory_temp}/{video_name}-(temp)-audio.mp4"

    draw_bbox_from_file(
        file_path=raw_txt_path,
        input_video_path=full_video_path,
        output_video_path=processed_with_audio,
        annotation_type="bbox",
        draw_frame_count=True,
    )

    if has_audio_stream(full_video_path):
        print("Adding audio...")
        mux_audio(full_video_path, processed_with_audio, processed_with_audio)
    else:
        print("No audio stream detected; skipping mux.")
    print(f"Treatment done: {full_video_path}.\n")

20241015 - 12h41-treated.mp4 ignored
20241019 - 13h28.mp4 ignored
loma_mt.txt automatically created in /home/ucl/ingi/trixen/ChimpRec/ChimpVideo/input/manual_annotations.



Tracking progress (loma_mt.mp4):   0%|          | 0/1162 [00:00<?, ?it/s]

Tracking progress (loma_mt.mp4): 100%|██████████| 1162/1162 [00:38<00:00, 29.98it/s] 


Annotations ready for video: /home/ucl/ingi/trixen/ChimpRec/ChimpVideo/input/loma_mt.mp4.



Drawing annotations (loma_mt.mp4): 100%|██████████| 1162/1162 [00:05<00:00, 201.24it/s]

Adding audio...
[mux_audio] ffmpeg not found (tried 'ffmpeg'); skipping audio mux.
Treatment done: /home/ucl/ingi/trixen/ChimpRec/ChimpVideo/input/loma_mt.mp4.

20241019 - 14h29.mp4 ignored


<h2>Second step:</h2>

This final step will take into account your modifications to modify the output of the automated process.

<b>If you need to modify annotations previously created:</b> simply run this part of the code. In this case, there's no need to run the above cells.

In [3]:
# ---------- STEP 2: apply manual edits -> treated output (per-video folder) ----------
for input_video in os.listdir(input_video_directory):
    if not (input_video.endswith(".mp4") or input_video.endswith(".MP4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S2:
        print(f"{video_name}.mp4 ignored")
        continue

    annotation_file = f"{mannual_annotations_directory}/{video_name}.txt"
    raw_reader = raw_tracking_data_reader(f"{raw_text_output_directory}/{video_name}.txt")

    try:
        edit_reader = modification_reader(annotation_file)
    except Exception:
        print(
            f"Error: the manual annotation file related to the video <{full_video_path}> is not found. "
            f"It must be located at <{annotation_file}>."
        )
        continue

    # per-video output folder
    video_out_dir = os.path.join(treated_directory, video_name)
    os.makedirs(video_out_dir, exist_ok=True)

    metadata_file_path = os.path.join(video_out_dir, f"{video_name}-treated.txt")
    output_video_path = os.path.join(video_out_dir, f"{video_name}-treated.mp4")
    writer = data_writer(metadata_file_path)

    modified_data = edit_raw_output(raw_reader, edit_reader)
    writer.write(modified_data)

    draw_bbox_from_file(
        file_path=metadata_file_path,
        input_video_path=full_video_path,
        output_video_path=output_video_path,
        annotation_type="bbox",
        draw_frame_count=True,
    )

    if has_audio_stream(full_video_path):
        print("Adding audio...")
        mux_audio(full_video_path, output_video_path, output_video_path)
    else:
        print("No audio stream detected; skipping mux.")
    print(f"Treatment done: {full_video_path}.\n")

20241015 - 12h41-treated.mp4 ignored
20241019 - 13h28.mp4 ignored


Drawing annotations (loma_mt.mp4):   0%|          | 0/1162 [00:00<?, ?it/s]

Drawing annotations (loma_mt.mp4): 100%|██████████| 1162/1162 [00:05<00:00, 222.85it/s]

Adding audio...
[mux_audio] ffmpeg not found (tried 'ffmpeg'); skipping audio mux.
Treatment done: /home/ucl/ingi/trixen/ChimpRec/ChimpVideo/input/loma_mt.mp4.

20241019 - 14h29.mp4 ignored


### Third step

In [ ]:
# ---------- STEP 3: final arrows/names -> final output (per-video folder) ----------
for input_video in os.listdir(input_video_directory):
    if not (input_video.endswith(".mp4") or input_video.endswith(".MP4")):
        continue

    full_video_path = os.path.join(input_video_directory, input_video)
    video_name = os.path.splitext(input_video)[0]

    if video_name in ignore_S3:
        print(f"{video_name}.mp4 ignored")
        continue

    annotation_file = f"{mannual_annotations_directory}/{video_name}.txt"
    raw_reader = raw_tracking_data_reader(f"{raw_text_output_directory}/{video_name}.txt")

    try:
        edit_reader = modification_reader(annotation_file)
    except Exception:
        print(
            f"Error: the manual annotation file related to the video <{full_video_path}> is not found. "
            f"It must be located at <{annotation_file}>."
        )
        continue

    # per-video output folder
    video_out_dir = os.path.join(final_directory, video_name)
    os.makedirs(video_out_dir, exist_ok=True)

    metadata_file_path = os.path.join(video_out_dir, f"{video_name}-final.txt")
    output_video_path = os.path.join(video_out_dir, f"{video_name}-final.mp4")
    writer = data_writer(metadata_file_path)

    modified_data = edit_raw_output(raw_reader, edit_reader)
    writer.write(modified_data)

    draw_bbox_from_file(
        file_path=metadata_file_path,
        input_video_path=full_video_path,
        output_video_path=output_video_path,
        annotation_type="triangle",
        draw_frame_count=False,
    )

    if has_audio_stream(full_video_path):
        print("Adding audio...")
        mux_audio(full_video_path, output_video_path, output_video_path)
    else:
        print("No audio stream detected; skipping mux.")
    print(f"Treatment done: {full_video_path}.\n")